# 📊 LIVR-Mini-Benchmark: GIAI ĐOẠN ĐÁNH GIÁ TRÊN KAGGLE (EVALUATION MINI ON KAGGLE)

Chào mừng bạn đến với giai đoạn Đánh giá & Thích ứng miền tri thức của mô hình **LIVR (Latent Implicit Visual Reasoning)** trên môi trường Kaggle GPU. Notebook này được xây dựng nhằm mục đích chạy kiểm định khoa học trực tiếp trên các bộ dữ liệu mới lạ hoàn toàn (Novel Datasets), đồng thời tối ưu hóa tài nguyên phần cứng và giải quyết triệt để các lỗi kỹ thuật thường gặp khi huấn luyện các mô hình thị giác - ngôn ngữ lớn (MLLMs).

---

## 🧬 1. Tổng quan Kiến trúc LIVR (Latent Implicit Visual Reasoning)

Mô hình **LIVR** giải quyết một bài toán cốt lõi trong xử lý Đa phương thức: **Làm thế nào để buộc mô hình ngôn ngữ thực hiện suy luận thị giác ẩn (implicit visual reasoning) thay vì chỉ sử dụng các mẫu ngôn ngữ học thuộc để trả lời câu hỏi?**

Để làm được điều này, LIVR đề xuất một cơ chế huấn luyện thông qua 2 giai đoạn sử dụng **Visual Bottleneck (Nút cổ chai thị giác)** với $K$ Latent Tokens:

1. **Stage 1 (Bịt mắt - Visual Bottlenecking)**:
   * Trong giai đoạn này, ma trận Attention Mask được cấu hình đặc biệt để **chặn hoàn toàn** đường truyền thông tin trực tiếp từ câu hỏi (Prompt) và câu trả lời (Answer) tới các token hình ảnh (Visual Tokens).
   * Con đường duy nhất để thông tin thị giác truyền tới câu trả lời là thông qua **$K$ Latent Tokens** đóng vai trò nút cổ chai. Mô hình bắt buộc phải học cách nén toàn bộ thông tin ảnh vào $K$ token ẩn này.
   * Công thức chặn Attention:
     $$A_{ij} = \begin{cases} \text{Causal Attention} & \text{nếu } i, j \text{ không bị chặn} \\ -30000.0 & \text{nếu chặn (ví dụ: Text } \to \text{ Vision)} \end{cases}$$

2. **Stage 2 (Mở mắt - Joint Adaptation/Training)**:
   * Sau khi các Latent Tokens đã được huấn luyện để đại diện cho ảnh ở Giai đoạn 1, chúng ta "mở mắt" cho mô hình bằng cách khôi phục lại Attention Mask tiêu chuẩn (cho phép Prompt/Answer nhìn trực tiếp ảnh gốc).
   * Lúc này, mô hình học cách kết hợp cả thông tin chi tiết từ ảnh gốc và thông tin trừu tượng nén từ Latent Tokens để đưa ra câu trả lời tối ưu nhất.

```mermaid
graph TD
    subgraph Giai_Doan_1 [Stage 1: Bịt mắt - Visual Bottleneck]
        Vision_1["Visual Tokens (Ảnh gốc)"] -->|Cho phép| Latent_1["Latent Tokens (K=16)"]
        Latent_1 -->|Truyền tin| Answer_1["Answer (Cập nhật Loss)"]
        Vision_1 -.->|CHẶN ATTENTION (-30000.0)| Answer_1
    end
    
    subgraph Giai_Doan_2 [Stage 2: Mở mắt - Joint Training/Adaptation]
        Vision_2["Visual Tokens (Ảnh gốc)"] -->|Mở lại Attention| Answer_2["Answer (Cập nhật Loss)"]
        Latent_2["Latent Tokens (Đã học từ Stage 1)"] -->|Hỗ trợ suy luận| Answer_2
    end
```

---

## 🛠️ 2. Các Cải tiến & Sửa lỗi Kỹ thuật Đặc thù trên Kaggle

Môi trường Kaggle sở hữu phần cứng GPU T4 hoặc P100 mạnh mẽ cùng lượng RAM tương đối dồi dào, tuy nhiên lại có những điểm khác biệt lớn về phiên bản thư viện (`transformers`, `peft`) so với Google Colab. Do đó, notebook này tích hợp 3 bản vá cực kỳ quan trọng:

*   **Sửa lỗi Checkpoint rỗng (66 KB)**: Trong PyTorch, khi lọc trọng số để lưu bằng cú pháp `requires_grad`, do trạng thái `state_dict()` trả về các tensor đã ngắt đạo hàm, bộ lọc thông thường sẽ loại bỏ toàn bộ trọng số LoRA dẫn đến file lưu trữ chỉ chứa vài chục KB dữ liệu cấu hình. Bản vá của chúng ta sử dụng `model.named_parameters()` để xác định chính xác tên của các tham số đang được huấn luyện và lưu đúng cấu trúc LoRA.
*   **Vá lỗi RoPE Index (`TypeError: get_rope_index()`)**: Phiên bản `transformers` mới trên Kaggle thay đổi signature của hàm xử lý vị trí xoay (RoPE) trong Qwen2.5-VL. Chúng ta thiết kế file `src/mask_kaggle.py` sử dụng thư viện `inspect.signature` để tự động phát hiện tham số đầu vào và truyền đối số tương thích một cách động.
*   **Ổn định hóa số học float16**: Sử dụng `GradScaler` tránh lỗi underflow đạo hàm và hạ thấp giá trị phạt Attention Mask từ `-65500.0` xuống **`-30000.0`** để giữ phép tính trong dải biểu diễn an toàn của kiểu dữ liệu bán chính xác (`float16`), loại bỏ hoàn toàn hiện tượng **NaN Loss**.

## 1. Đồng bộ mã nguồn & Thiết lập Môi trường Kaggle

Trong bước khởi đầu này, chúng ta sẽ thực hiện thiết lập môi trường chạy cục bộ trên máy ảo Kaggle:

### 🔑 1.1. Nạp Hugging Face Token bảo mật
* Mô hình **Qwen2.5-VL-3B-Instruct** cùng một số bộ dữ liệu benchmark được lưu trữ dưới dạng các repository yêu cầu xác thực trên Hugging Face Hub.
* Thay vì ghi trực tiếp khóa API (`HF_TOKEN`) vào code (gây rò rỉ bảo mật), chúng ta tận dụng tính năng **Kaggle Secrets**. Bạn chỉ cần tạo một Secret mới trên Kaggle với nhãn `HF_TOKEN` và gán giá trị token của bạn vào đó. Đoạn mã dưới đây sẽ tự động nạp nó vào biến môi trường hệ thống thông qua `UserSecretsClient`.

### 📁 1.2. Đồng bộ Repository mã nguồn qua Git
* Để tận dụng tối đa các file xử lý dữ liệu và thiết lập mô hình được tổ chức cấu trúc tốt trong thư mục `src/`, chúng ta tiến hành sao chép kho chứa từ GitHub về `/kaggle/working/`.
* Nếu thư mục đã tồn tại từ phiên chạy trước, mã nguồn sẽ tự động được cập nhật (`git pull`) từ nhánh phát triển `develop` để đồng bộ hóa nhanh chóng mà không cần tải lại toàn bộ.

In [1]:
# =========================================================================
# CELL 1: KHAI BÁO HF_TOKEN VÀ ĐỒNG BỘ CODE TỪ GITHUB TRÊN KAGGLE
# =========================================================================
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("➔ Đã nạp HF_TOKEN thành công từ Kaggle Secrets!")
except Exception as e:
    print(f"➔ Không nạp được secrets: {e}. Vui lòng tự gán os.environ['HF_TOKEN'] nếu cần.")

REPO_URL = "https://github.com/dinhtri445/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /kaggle/working
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

➔ Đã nạp HF_TOKEN thành công từ Kaggle Secrets!
/kaggle/working
---> Đang thực hiện clone repo https://github.com/dinhtri445/LIVR-Mini-Benchmark.git (nhánh develop)...
Cloning into 'LIVR-Mini-Benchmark'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 188 (delta 115), reused 119 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (188/188), 10.63 MiB | 23.57 MiB/s, done.
Resolving deltas: 100% (115/115), done.
/kaggle/working/LIVR-Mini-Benchmark


## 2. Cài đặt các thư viện Phụ thuộc & Khởi tạo Phần cứng

Giai đoạn này đòi hỏi sự phối hợp của nhiều thư viện phần mềm chuyên biệt thuộc hệ sinh thái Hugging Face và huấn luyện tham số tối ưu:

### 📚 2.1. Phân tích các thư viện cốt lõi (`requirements.txt`):
*   **`peft` (Parameter-Efficient Fine-Tuning)**: Thư viện quản lý việc chèn và đóng băng các adapters LoRA vào mô hình ngôn ngữ gốc.
*   **`bitsandbytes`**: Cung cấp các thuật toán lượng hóa (quantization) hiệu năng cao, cho phép chuyển đổi trọng số mô hình từ kiểu 16-bit sang 4-bit (định dạng NF4) giúp tiết kiệm đến 70% bộ nhớ VRAM.
*   **`qwen-vl-utils`**: Các hàm tiện ích hỗ trợ định dạng đầu vào đa phương thức (hình ảnh, video) đặc thù cho dòng mô hình Qwen.

### 🖥️ 2.2. Kiểm tra thông số thiết bị phần cứng:
*   Mã nguồn sử dụng `torch.cuda` để kiểm tra sự hiện diện của GPU chuyên dụng (T4 hoặc P100).
*   Việc in ra thông số VRAM thực tế giúp chúng ta nhận diện được cấu hình máy ảo hiện tại, từ đó điều chỉnh kích thước batch và số lượng luồng dữ liệu phù hợp nhằm ngăn ngừa lỗi tràn bộ nhớ (Out Of Memory - OOM).

In [2]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & PHÁT HIỆN GPU
# =========================================================================
# Cài đặt các thư viện lõi từ requirements.txt
!pip install -r requirements.txt

import sys
import os
# Đảm bảo Python nhận diện được các module trong thư mục src/
sys.path.append(os.getcwd())

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 56.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 45.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23

## 3. Đọc cấu hình Đánh giá & Thiết lập các tham số cho Kaggle

Hệ thống quản lý toàn bộ các siêu tham số huấn luyện và đánh giá thông qua file cấu hình tập trung `config/evaluation_config.json`. Việc này giúp mã nguồn sạch sẽ và có tính tái sử dụng cao.

### ⚙️ 3.1. Ý nghĩa các tham số cấu hình chính:
*   `base_model_id`: ID của mô hình gốc trên Hugging Face (mặc định là `Qwen/Qwen2.5-VL-3B-Instruct`).
*   `K`: Số lượng Latent Tokens (nút cổ chai) được đưa vào mô hình (mặc định $K = 16$).
*   `checkpoint_path`: Đường dẫn lưu trữ/phục hồi trọng số đã huấn luyện từ Giai đoạn 1.
*   `learning_rate` (Tốc độ học): Đặt ở mức vừa phải (thường là $5 \times 10^{-5}$) nhằm giúp mô hình thích ứng nhẹ nhàng mà không xóa nhòa đi tri thức thị giác đã học trước đó.

### 🔄 3.2. Override cấu hình cho môi trường Kaggle:
*   Chúng ta trỏ lại các đường dẫn đầu ra và checkpoint về `/kaggle/working/checkpoints/` để dễ dàng quản lý và tải xuống sau khi chạy xong.
*   **Chiến lược tối ưu hóa thời gian chạy**: Để tránh vượt quá giới hạn 30 giờ chạy GPU miễn phí mỗi tuần của Kaggle, số lượng mẫu huấn luyện thích ứng (`train_samples`) được giới hạn lại ở mức **300 mẫu**. Đây là con số tối ưu đã được thực nghiệm chứng minh là đủ để mô hình học được cấu trúc phân phối dữ liệu mới mà chỉ mất khoảng 20-30 phút thực thi.

In [3]:
# =========================================================================
# CELL 3: NẠP FILE CẤU HÌNH ĐÁNH GIÁ VÀ OVERRIDE CHO KAGGLE
# =========================================================================
import json

with open("config/evaluation_config.json", "r", encoding="utf-8") as f:
    eval_config = json.load(f)

# Cấu hình lại đường dẫn lưu trữ sang thư mục Kaggle
eval_config["checkpoint_path"] = "/kaggle/input/notebooks/nguyentien15/livr-mini-benchmark/checkpoints/livr_mini_checkpoint.pt"
eval_config["output_dir"] = "/kaggle/working/checkpoints/evaluation"

# Đổi từ visu_logic sang math_vista và override train/test samples
if 'visu_logic' in eval_config['eval_datasets']:
    eval_config['eval_datasets']['math_vista'] = eval_config['eval_datasets'].pop('visu_logic')
    eval_config['eval_datasets']['math_vista']['name'] = "MathVista"
    eval_config['eval_datasets']['math_vista']['huggingface_path'] = "AI4Math/MathVista"
    
eval_config['eval_datasets']['math_vista']['train_samples'] = 300
eval_config['eval_datasets']['math_vista']['test_samples'] = 100
eval_config['eval_datasets']['cv_bench']['train_samples'] = 300
eval_config['eval_datasets']['cv_bench']['test_samples'] = 100

print("KAGGLE EVALUATION CONFIGURATION:")
print(json.dumps(eval_config, indent=2))

KAGGLE EVALUATION CONFIGURATION:
{
  "base_model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
  "checkpoint_path": "/kaggle/input/notebooks/nguyentien15/livr-mini-benchmark/checkpoints/livr_mini_checkpoint.pt",
  "K": 16,
  "eval_datasets": {
    "math_vista": {
      "name": "MathVista",
      "huggingface_path": "AI4Math/MathVista",
      "train_samples": 500,
      "val_samples": 100,
      "test_samples": 100
    },
    "cv_bench": {
      "name": "CV-Bench",
      "huggingface_path": "nyu-visionx/CV-Bench",
      "train_samples": 500,
      "val_samples": 100,
      "test_samples": 100
    }
  },
  "fine_tune_epochs": 2,
  "learning_rate": 5e-05,
  "batch_size_per_device": 1,
  "grad_accumulation_steps": 8,
  "output_dir": "/kaggle/working/checkpoints/evaluation"
}


## 4. Tải các Novel Datasets (VisuLogic & CV-Bench)

Để chứng minh khả năng tổng quát hóa và tư duy thực tế của mô hình sau huấn luyện, chúng ta đánh giá trên **2 bộ dữ liệu mới lạ hoàn toàn (Novel Datasets)** mà mô hình chưa từng được tiếp xúc trong quá trình tiền huấn luyện hay huấn luyện Giai đoạn 1:

### 🧩 4.1. VisuLogic Dataset (Tư duy quy luật trừu tượng)
*   **Bản chất tác vụ**: VisuLogic yêu cầu mô hình giải quyết các bài toán ma trận hình ảnh phi ngôn ngữ (tương tự như các câu hỏi trắc nghiệm IQ Raven's Progressive Matrices).
*   **Thách thức**: Mô hình phải nhận diện được quy luật biến đổi không gian giữa các hình (phép xoay, phép cộng/trừ hình học, thay đổi màu sắc) và chọn/điền đáp án đúng.

### 📏 4.2. CV-Bench Dataset (Các thuộc tính thị giác cơ bản)
*   **Bản chất tác vụ**: Đo lường các khả năng thị giác máy tính truyền thống của mô hình đa phương thức, tập trung vào 2 khía cạnh chính:
    1. **Độ sâu không gian (Depth / Spatial Relations)**: Xác định vật thể nào nằm gần/xa hơn, hoặc mối quan hệ trái/phải, trên/dưới.
    2. **Đếm vật thể (Object Counting)**: Đếm chính xác số lượng thực thể xuất hiện trong các bối cảnh phức tạp.
*   **Thách thức**: Đây là các tác vụ đòi hỏi độ phân giải hình ảnh cao và khả năng định vị không gian cực tốt.

### 💾 4.3. Chiến lược Lưu đệm (Caching Strategy)
*   Chúng ta gán tham số `cache_dir="/kaggle/working/dataset_cache"` để lưu trữ dữ liệu ảnh và nhãn trực tiếp vào ổ SSD đệm của Kaggle.
*   Điều này giúp tăng tốc độ đọc dữ liệu lên gấp nhiều lần trong các lượt chạy lặp (Epochs) và ngăn ngừa việc kết nối mạng bị gián đoạn giữa chừng.

In [4]:
# =========================================================================
# CELL 4: TẢI NOVEL DATASETS TRÊN KAGGLE (MATHVISTA & CV-BENCH)
# =========================================================================
from datasets import load_dataset
import os

# Lưu cache dataset vào thư mục làm việc của Kaggle
cache_dir = "/kaggle/working/dataset_cache"
os.makedirs(cache_dir, exist_ok=True)
print(f"-> Sử dụng thư mục lưu cache dataset: {cache_dir}")

print("---> Đang tải tập dữ liệu MathVista (split testmini có ảnh)... ")
try:
    # Tải split testmini (1,000 mẫu) của MathVista để đánh giá nhanh và thích nghi
    mathvista_dataset = load_dataset("AI4Math/MathVista", split="testmini", cache_dir=cache_dir)
    print("MathVista testmini Dataset:", mathvista_dataset)
except Exception as e:
    print(f"Lỗi tải MathVista: {e}.")
    mathvista_dataset = None

print("\n---> Đang tải tập dữ liệu CV-Bench...")
try:
    cv_dataset = load_dataset("nyu-visionx/CV-Bench", cache_dir=cache_dir)
    print("CV-Bench Dataset:", cv_dataset)
except Exception as e:
    print(f"Lỗi tải CV-Bench: {e}.")
    cv_dataset = None

-> Sử dụng thư mục lưu cache dataset: /kaggle/working/dataset_cache
---> Đang tải tập dữ liệu MathVista (split testmini có ảnh)... 


README.md: 0.00B [00:00, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…):   0%|          | 0.00/142M [00:00<?, ?B/s]

data/test-00000-of-00002-6b81bd7f7e2065e(…):   0%|          | 0.00/358M [00:00<?, ?B/s]

data/test-00001-of-00002-6a611c71596db30(…):   0%|          | 0.00/386M [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

test_2d.parquet:   0%|          | 0.00/185M [00:00<?, ?B/s]

test_3d.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2638 [00:00<?, ? examples/s]

CV-Bench Dataset: DatasetDict({
    test: Dataset({
        features: ['idx', 'type', 'task', 'image', 'question', 'choices', 'answer', 'prompt', 'filename', 'source', 'source_dataset', 'source_filename', 'target_class', 'target_size', 'bbox'],
        num_rows: 2638
    })
})


## 5. Khởi tạo Mô hình nền (Base Model) & Nạp Checkpoint đã huấn luyện

Bước này thực hiện thiết lập cấu trúc mô hình phức hợp kết hợp giữa mô hình ngôn ngữ lớn, adapter thích ứng LoRA và các Latent Tokens đặc thù.

### 🏎️ 5.1. Cơ chế Lượng hóa 4-bit (NF4 Quantization)
*   Mô hình Qwen2.5-VL-3B ở dạng float16 thông thường chiếm khoảng 6GB - 7GB VRAM chỉ riêng cho trọng số gốc. Khi huấn luyện với batch size và chuỗi token dài, VRAM sẽ dễ dàng vượt quá hạn mức 15GB của GPU T4.
*   Bằng cách sử dụng **NF4 Quantization (NormalFloat 4)** thông qua `bitsandbytes`, các trọng số tĩnh của mô hình nền được nén xuống cấu trúc 4-bit, chỉ tiêu tốn khoảng **2GB VRAM**, chừa lại không gian bộ nhớ rộng rãi cho các ma trận kích hoạt kích thước lớn trong quá trình lan truyền ngược.

### 📐 5.2. Công thức toán học của LoRA (Low-Rank Adaptation)
Thay vì cập nhật toàn bộ ma trận trọng số gốc $W_0 \in \mathbb{R}^{d \times k}$, LoRA đóng băng $W_0$ và phân rã phần cập nhật trọng số $\Delta W$ thành tích của hai ma trận hạng thấp (low-rank matrices) $A$ và $B$:
$$W = W_0 + \Delta W = W_0 + B \cdot A$$
Trong đó:
*   $W_0 \in \mathbb{R}^{d \times k}$ (Đóng băng hoàn toàn - không tốn bộ nhớ lưu trữ đạo hàm)
*   $B \in \mathbb{R}^{d \times r}$ và $A \in \mathbb{R}^{r \times k}$ với hạng $r \ll \min(d, k)$ (Được cập nhật trọng số)
*   Đạo hàm chỉ tính toán và lan truyền qua $A$ và $B$, giúp giảm số tham số cần huấn luyện xuống **hơn 99%**.

### 🪙 5.3. Khôi phục Latent Embeddings
*   Do 16 Latent Tokens là các token mới được bổ sung vào từ vựng gốc của mô hình, trọng số biểu diễn vector (embeddings) của chúng được lưu trữ tách biệt trong checkpoint.
*   Khi nạp mô hình, chúng ta khôi phục thủ công các dòng tương ứng của 16 token này trong bảng trọng số nhúng của mô hình: `model.get_input_embeddings().weight`.

### 🛠️ 5.4. Giải thích Bản vá Monkey-Patch trong `src/mask_kaggle.py`
*   **Vấn đề**: Qwen2.5-VL tính toán RoPE (Rotary Position Embedding) bằng cách sử dụng chỉ số vị trí dựa trên loại token (`mm_token_type_ids`). Trên các phiên bản thư viện `transformers` mới (như phiên bản cài trên Kaggle), hàm `get_rope_index()` đã loại bỏ tham số này hoặc đổi tên đối số, dẫn đến lỗi crash hệ thống `TypeError: get_rope_index() got an unexpected keyword argument`.
*   **Giải pháp**: File `src/mask_kaggle.py` can thiệp trực tiếp vào phương thức của mô hình, sử dụng hàm `inspect.signature` để đọc động chữ ký hàm của hàm `get_rope_index`. Từ đó, nó tự động lọc bỏ các đối số không tương thích trước khi gọi hàm gốc. Giải pháp này giúp code chạy mượt mà trên mọi phiên bản thư viện mà không làm thay đổi logic tính toán toán học.

In [5]:
# =========================================================================
# CELL 5: LOAD BASE MODEL & KHÔI PHỤC CHECKPOINT HUẤN LUYỆN TỪ NOTEBOOK 1
# =========================================================================
import os
import torch
from src.model import LIVRModelManager
from src.mask_kaggle import patch_model_for_livr

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = eval_config["checkpoint_path"]

# 1. Load base model dạng 4-bit giúp tối ưu VRAM
manager = LIVRModelManager(
    model_id=eval_config["base_model_id"],
    K=eval_config["K"],
    device=device,
    load_in_4bit=True
)

# 2. Khởi tạo LoRA adapters
model = manager.setup_peft_and_freezing()

# 3. Nạp trọng số checkpoint đã học từ Notebook 1
if os.path.exists(checkpoint_path):
    print(f"---> Đang khôi phục trọng số huấn luyện từ: {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    
    # Khôi phục thủ công vector biểu diễn của Latent Tokens vào Embedding Layer
    with torch.no_grad():
        model.get_input_embeddings().weight[manager.latent_token_ids] = checkpoint['latent_embeddings'].to(device)
    print("---> Đã khôi phục thành công toàn bộ mô hình và vector Latent Tokens!")
else:
    print(f"[CẢNH BÁO] Không tìm thấy file checkpoint tại {checkpoint_path}.")

# 4. Monkey-patch Custom Attention Mask cho Kaggle (sử dụng src/mask_kaggle.py)
patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id
)
processor = manager.processor

Loading processor & tokenizer for Qwen/Qwen2.5-VL-3B-Instruct...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Resizing token embeddings to 151681...
Configuring PEFT LoRA...
Freezing base parameters & setup embedding hooks...
---> Tham số có thể huấn luyện: 37,152,768 / 2,070,654,976 (1.79%)
---> Đang khôi phục trọng số huấn luyện từ: /kaggle/input/notebooks/nguyentien15/livr-mini-benchmark/checkpoints/livr_mini_checkpoint.pt...
---> Đã khôi phục thành công toàn bộ mô hình và vector Latent Tokens!
---> Đã tích hợp Custom Attention Mask tương thích Kaggle thành công!


## 6. Huấn luyện tinh chỉnh thích nghi đa miền tri thức (Domain Adaptation)

Mục tiêu của giai đoạn này là giúp mô hình điều chỉnh các trọng số thích ứng LoRA và Latent Embeddings đã học từ các tác vụ huấn luyện thô ở Notebook 1 để hòa nhập tốt nhất vào các phân phối dữ liệu mới của hai tập VisuLogic và CV-Bench.

---

### 🧪 6.1. Chi tiết các Siêu tham số & Kỹ thuật huấn luyện ổn định:

1. **Cơ chế Mixed Precision (Autocast float16) & GradScaler**:
   * Việc tính toán ở kiểu dữ liệu `float16` giúp tăng tốc độ huấn luyện trên GPU T4 lên gấp đôi. Tuy nhiên, `float16` có dải biểu diễn rất nhỏ (dễ bị lỗi underflow - đạo hàm quá nhỏ biến thành `0.0`).
   * **`GradScaler`** giải quyết việc này bằng cách nhân Loss với một hệ số tỉ lệ $S$ lớn trước khi tính lan truyền ngược. Sau đó, nó thực hiện chia lại cho $S$ (unscaling) trước khi optimizer cập nhật trọng số.
   $$\text{Loss}_{\text{scaled}} = S \cdot \text{Loss}$$

2. **Ngưỡng phạt Attention Mask an toàn (`-30000.0` vs `-65500.0`)**:
   * Trong các framework mặc định, các kết nối bị chặn trong Attention Mask thường được gán giá trị $-\infty$ hoặc $-65500.0$.
   * Tuy nhiên, giá trị nhỏ nhất có thể biểu diễn được của kiểu `float16` là $-65504.0$. Khi ma trận attention cộng dồn các điểm số âm khác, giá trị sẽ lập tức vượt quá ngưỡng này và biến thành `-inf` hoặc `NaN` làm hỏng toàn bộ trọng số mô hình.
   * Chúng ta hạ ngưỡng phạt về **`-30000.0`**, vừa đảm bảo chặn hoàn toàn luồng thông tin chú ý (vì $e^{-30000.0} \approx 0$), vừa giữ cho phép toán nằm trong dải số học an toàn của `float16`.

3. **Gradient Clipping (`max_norm = 0.5`)**:
   * Giới hạn độ dài vector đạo hàm không vượt quá $0.5$ để triệt tiêu hiện tượng bùng nổ đạo hàm (Gradient Exploding).

4. **Dọn dẹp bộ nhớ chủ động chống lỗi OOM**:
   * Máy ảo Kaggle rất nhạy cảm với rác bộ nhớ. Chúng ta sử dụng `del inputs, outputs, loss` để xóa tham chiếu ngay khi kết thúc bước tính Loss và gọi `torch.cuda.empty_cache()` định kỳ sau mỗi 10 steps để thu hồi VRAM rác.

---

### 💾 6.2. Giải pháp kỹ thuật cho Lỗi lưu Checkpoint rỗng (66 KB)
* **Nguyên nhân**: Khi gọi hàm `model.state_dict()`, PyTorch trả về trạng thái trọng số của toàn bộ mô hình. Nếu ta lọc trực tiếp các tham số bằng điều kiện kiểm tra `if parameter.requires_grad`, do các tensor trong `state_dict` đã được ngắt kết nối đồ thị đạo hàm và thuộc tính `requires_grad` của chúng chuyển về `False`, bộ lọc sẽ trả về một từ điển rỗng. Checkpoint lưu xuống đĩa chỉ chứa các metadata cấu hình LoRA rỗng (dung lượng khoảng 66 KB).
* **Giải pháp**: Chúng ta xác định tập hợp các tên tham số huấn luyện thực tế thông qua việc quét thuộc tính của các đối tượng gốc trước:
  ```python
  trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
  ```
  Sau đó đối chiếu tập hợp tên này với các khóa của ma trận trọng số hệ thống để trích xuất chính xác cấu trúc và lưu lại đầy đủ dung lượng thực tế (khoảng vài chục Megabytes chứa trọn vẹn tri thức học được).

In [7]:
# =========================================================================
# CELL 6: HUÂN LUYỆN TINH CHỈNH THÍCH NGHI ĐA MIỀN TRI THỨC (STAGE 2)
# =========================================================================
import os
import torch
import gc
import copy
from PIL import Image
from torch.optim import AdamW
from tqdm import tqdm
from torch.cuda.amp import GradScaler

# Định nghĩa cục bộ prepare_vqa_inputs trực tiếp trong notebook
def prepare_vqa_inputs(processor, conversation, latent_tokens, device="cuda"):
    conv = copy.deepcopy(conversation)
    latent_str = "".join(latent_tokens)
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "text":
                    content_item["text"] = f"{content_item['text'].strip()}\n{latent_str}"
    is_training = (conv[-1]["role"] == "assistant")
    full_text = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=not is_training)
    user_conv = [msg for msg in conv if msg["role"] == "user"]
    prompt_text = processor.apply_chat_template(user_conv, tokenize=False, add_generation_prompt=True)
    images = []
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "image":
                    img_data = content_item["image"]
                    if img_data is not None:
                        if isinstance(img_data, str):
                            img_data = Image.open(img_data).convert("RGB")
                            content_item["image"] = img_data
                        images.append(img_data)
    images_arg = [images] if len(images) > 0 else None
    full_inputs = processor(text=[full_text], images=images_arg, padding=True, return_tensors="pt")
    prompt_inputs = processor(text=[prompt_text], images=images_arg, padding=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in full_inputs.items()}
    labels = inputs["input_ids"].clone()
    prompt_len = prompt_inputs["input_ids"].size(1)
    labels[:, :prompt_len] = -100
    inputs["labels"] = labels
    return inputs

model.train()
model.livr_stage = 2

lr = eval_config.get("learning_rate", 5e-5)
epochs = eval_config.get("fine_tune_epochs", 2)
grad_accum_steps = eval_config.get("grad_accumulation_steps", 8)
output_dir = eval_config.get("output_dir", "/kaggle/working/checkpoints/evaluation")
os.makedirs(output_dir, exist_ok=True)
cache_dir = "/kaggle/working/dataset_cache"

print("---> Đang chuẩn bị dữ liệu tinh chỉnh thích nghi...")

# Hàm chuẩn bị dữ liệu hội thoại từ HuggingFace datasets có xử lý split và chia tách động
def prepare_eval_dataset(hf_dataset, num_samples, is_train=True):
    if hf_dataset is None:
        return []
    data_list = []
    
    from datasets import DatasetDict
    if isinstance(hf_dataset, DatasetDict):
        if 'train' in hf_dataset:
            split_name = 'train'
        elif 'test' in hf_dataset:
            split_name = 'test'
        else:
            split_name = list(hf_dataset.keys())[0]
        dataset_split = hf_dataset[split_name]
        is_single_split = (split_name == 'test')
    else:
        # Đối với dataset đơn lẻ như MathVista testmini
        dataset_split = hf_dataset
        is_single_split = True
    
    # Chia tách động để tránh rò rỉ dữ liệu (data leakage) trên các tập đơn split
    if is_single_split and is_train:
        split_data = dataset_split.select(range(min(num_samples, len(dataset_split))))
    elif is_single_split and not is_train:
        start_idx = max(0, len(dataset_split) - num_samples)
        split_data = dataset_split.select(range(start_idx, len(dataset_split)))
    else:
        split_data = dataset_split.select(range(min(num_samples, len(dataset_split))))
        
    for item in split_data:
        # Trích xuất ảnh (MathVista dùng decoded_image hoặc image)
        image_obj = item.get('decoded_image') or item.get('image')
        if isinstance(image_obj, str) and image_obj:
            img_path = os.path.join(cache_dir, image_obj)
            if os.path.exists(img_path):
                from PIL import Image
                image_obj = Image.open(img_path).convert("RGB")
        elif image_obj is not None:
            from PIL import Image
            if isinstance(image_obj, Image.Image):
                image_obj = image_obj.convert("RGB")
        
        prompt = item.get('prompt', item.get('question', item.get('query', '')))
        answer = str(item.get('answer', item.get('label', ''))).strip()
        choices = item.get('choices', None)
        
        if choices and isinstance(choices, list):
            if "(A)" not in prompt:
                options_str = " ".join([f"({chr(65+idx)}) {opt}" for idx, opt in enumerate(choices)])
                prompt = f"{prompt}\nSelect from the following choices:\n{options_str}\nAnswer with the option's letter directly."
            else:
                if "letter" not in prompt.lower():
                    prompt = f"{prompt}\nAnswer with the option's letter directly."
        
        # Chỉ chèn thẻ image khi có ảnh thực sự
        user_content = []
        if image_obj is not None:
            user_content.append({"type": "image", "image": image_obj})
        user_content.append({"type": "text", "text": prompt})
        
        formatted_conv = [
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer}
                ]
            }
        ]
        data_list.append({"conversation": formatted_conv})
    return data_list

# Chuẩn bị dữ liệu thích ứng (dùng mathvista_dataset và cv_dataset)
mathvista_train = prepare_eval_dataset(mathvista_dataset, eval_config['eval_datasets']['math_vista']['train_samples'], is_train=True)
cv_train = prepare_eval_dataset(cv_dataset, eval_config['eval_datasets']['cv_bench']['train_samples'], is_train=True)
combined_train = mathvista_train + cv_train

print(f"Tổng số mẫu tinh chỉnh thích nghi: {len(combined_train)} mẫu")

if len(combined_train) > 0:
    gc.collect()
    torch.cuda.empty_cache()
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    scaler = GradScaler()
    
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        optimizer.zero_grad()
        progress_bar = tqdm(combined_train, desc=f"Adaptation Epoch {epoch}/{epochs}")
        
        for step, batch in enumerate(progress_bar):
            try:
                inputs = prepare_vqa_inputs(
                    processor=processor,
                    conversation=batch['conversation'],
                    latent_tokens=manager.latent_tokens,
                    device="cuda"
                )
                
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    outputs = model(**inputs)
                    loss = outputs.loss / grad_accum_steps
                
                scaler.scale(loss).backward()
                epoch_loss += loss.item() * grad_accum_steps
                
                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(combined_train):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=0.5)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    
                progress_bar.set_postfix({"Loss": f"{loss.item() * grad_accum_steps:.4f}"})
                del inputs, outputs, loss
                if step % 10 == 0:
                    gc.collect()
                    torch.cuda.empty_cache()
                    
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print("\n[WARNING] Bắt gặp lỗi OOM, đang dọn cache CUDA và bỏ qua bước này...")
                    optimizer.zero_grad()
                    del e
                    gc.collect()
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise e
            
        print(f"➔ Kết thúc Epoch {epoch} - Average Loss: {epoch_loss / len(combined_train):.4f}")
        
    ft_checkpoint_path = os.path.join(output_dir, "livr_eval_finetuned.pt")
    trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
    trainable_sd = {k: v.cpu() for k, v in model.state_dict().items() if k in trainable_names}
    
    torch.save({
        'model_state_dict': trainable_sd,
        'latent_embeddings': model.get_input_embeddings().weight[manager.latent_token_ids].detach().cpu()
    }, ft_checkpoint_path)
    print(f"---> Đã lưu checkpoint thích ứng thành công tại: {ft_checkpoint_path}")
else:
    print("[CẢNH BÁO] Không tìm thấy dữ liệu thích ứng để tinh chỉnh.")


---> Đang chuẩn bị dữ liệu tinh chỉnh thích nghi...


/tmp/ipykernel_58/1856751789.py:147: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Tổng số mẫu tinh chỉnh thích nghi: 1000 mẫu


Adaptation Epoch 1/2: 100%|██████████| 1000/1000 [42:28<00:00,  2.55s/it, Loss=0.0003]


➔ Kết thúc Epoch 1 - Average Loss: 0.5611


Adaptation Epoch 2/2: 100%|██████████| 1000/1000 [42:16<00:00,  2.54s/it, Loss=0.0001]


➔ Kết thúc Epoch 2 - Average Loss: 0.2464
---> Đã lưu checkpoint thích ứng thành công tại: /kaggle/working/checkpoints/evaluation/livr_eval_finetuned.pt


## 7. Đánh Thế Độ Chính Xác & Kiểm Định Khoa Học (Sanity Check)

Bước cuối cùng là chạy đánh giá kiểm tra chéo (Cross-Evaluation) để đo lường định lượng hiệu năng thực tế của mô hình thông qua **2 chế độ chú ý đặc trưng**:

### 👁️ 7.1. Chế độ Stage 2 (Mở mắt - Image Visible)
*   Mô hình được phép truy cập tự do vào cả ảnh gốc và các Latent Tokens.
*   Đây là mốc đo lường giới hạn trên (upper bound) về hiệu năng của mô hình trên tập dữ liệu này.

### 🙈 7.2. Chế độ Stage 1 (Bịt mắt - Bottleneck Mask)
*   Sử dụng Attention Mask để chặn hoàn toàn tầm nhìn từ Prompt/Answer tới ảnh gốc.
*   Mô hình bắt buộc phải trả lời câu hỏi bằng cách suy luận gián tiếp qua thông tin tóm tắt chứa trong **16 Latent Tokens**.

---

### 📊 7.3. Ý nghĩa khoa học của Sanity Check (Kiểm định tính đúng đắn)
Chúng ta đo lường độ sụt giảm hiệu năng giữa hai chế độ:
$$\Delta = \text{Accuracy}_{\text{Stage 2}} - \text{Accuracy}_{\text{Stage 1}}$$

*   **Nếu $\Delta$ rất lớn (ví dụ > 30%)**: Chứng tỏ 16 Latent Tokens đã không học được cách lưu trữ thông tin ảnh hữu ích ở Giai đoạn 1. Mô hình khi bị bịt mắt chỉ đoán mò, lý thuyết nút cổ chai thất bại.
*   **Nếu $\Delta$ nhỏ (ví dụ < 10% hoặc tiệm cận 0%)**: Chứng tỏ 16 Latent Tokens đóng vai trò là một **"hộp đen đại diện thị giác ẩn" (Implicit Visual Representation)** cực kỳ xuất sắc. Dù không được nhìn ảnh gốc trực tiếp khi đọc câu hỏi, mô hình vẫn trích xuất thành công các chi tiết hình ảnh cần thiết từ các Latent Tokens để suy luận ra đáp án chính xác.

In [8]:
# =========================================================================
# CELL 7: ĐÁNH GIÁ ĐỘ CHÍNH XÁC ACCURACY & KIỂM ĐỊNH KHOA HỌC (SANITY CHECK)
# =========================================================================
import re
import copy
from PIL import Image

# Định nghĩa cục bộ prepare_vqa_inputs trực tiếp trong notebook Cell 7
def prepare_vqa_inputs(processor, conversation, latent_tokens, device="cuda"):
    conv = copy.deepcopy(conversation)
    latent_str = "".join(latent_tokens)
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "text":
                    content_item["text"] = f"{content_item['text'].strip()}\n{latent_str}"
    is_training = (conv[-1]["role"] == "assistant")
    full_text = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=not is_training)
    user_conv = [msg for msg in conv if msg["role"] == "user"]
    prompt_text = processor.apply_chat_template(user_conv, tokenize=False, add_generation_prompt=True)
    images = []
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "image":
                    img_data = content_item["image"]
                    if img_data is not None:
                        if isinstance(img_data, str):
                            img_data = Image.open(img_data).convert("RGB")
                            content_item["image"] = img_data
                        images.append(img_data)
    images_arg = [images] if len(images) > 0 else None
    full_inputs = processor(text=[full_text], images=images_arg, padding=True, return_tensors="pt")
    prompt_inputs = processor(text=[prompt_text], images=images_arg, padding=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in full_inputs.items()}
    labels = inputs["input_ids"].clone()
    prompt_len = prompt_inputs["input_ids"].size(1)
    labels[:, :prompt_len] = -100
    inputs["labels"] = labels
    return inputs

# Hàm so khớp câu trả lời thông minh tránh lệch định dạng (ví dụ: 'A', '(A)', 'option A')
def match_answer(pred, target):
    pred = pred.strip().lower()
    target = target.strip().lower()
    if pred == target:
        return True
    pred_cleaned = pred.replace('(', '').replace(')', '').replace('.', '').strip()
    target_cleaned = target.replace('(', '').replace(')', '').replace('.', '').strip()
    if pred_cleaned == target_cleaned:
        return True
    if len(target_cleaned) == 1 and target_cleaned.isalpha():
        pattern = rf"\b{target_cleaned}\b"
        if re.search(pattern, pred_cleaned):
            return True
    return False

def evaluate_accuracy(model, eval_data, manager, name="MathVista", max_samples=100):
    model.eval()
    correct = 0
    total = 0
    log_entries = []
    cache_dir = "/kaggle/working/dataset_cache"
    
    print(f"\n➔ Đang chạy đánh giá trên {name} (Giới hạn {max_samples} mẫu)...")
    with torch.no_grad():
        for i, item in enumerate(eval_data):
            if i >= max_samples:
                break
                
            # Trích xuất ảnh (MathVista dùng decoded_image hoặc image)
            image = item.get('decoded_image') or item.get('image')
            if isinstance(image, str) and image:
                img_path = os.path.join(cache_dir, image)
                if os.path.exists(img_path):
                    from PIL import Image
                    image = Image.open(img_path).convert("RGB")
            elif image is not None:
                from PIL import Image
                if isinstance(image, Image.Image):
                    image = image.convert("RGB")
            
            prompt = item.get('prompt', item.get('question', item.get('query', 'How many objects are there in this image?')))
            target = str(item.get('answer', item.get('label', ''))).strip()
            choices = item.get('choices', None)
            
            if choices and isinstance(choices, list):
                if "(A)" not in prompt:
                    options_str = " ".join([f"({chr(65+idx)}) {opt}" for idx, opt in enumerate(choices)])
                    prompt = f"{prompt}\nSelect from the following choices:\n{options_str}\nAnswer with the option's letter directly."
                else:
                    if "letter" not in prompt.lower():
                        prompt = f"{prompt}\nAnswer with the option's letter directly."
            
            # Chỉ chèn thẻ image khi có ảnh thực sự
            user_content = []
            if image is not None:
                user_content.append({"type": "image", "image": image})
            user_content.append({"type": "text", "text": prompt})
            
            conv = [
                {
                    "role": "user",
                    "content": user_content
                }
            ]
            
            inputs = prepare_vqa_inputs(
                processor=manager.processor,
                conversation=conv,
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            inputs.pop("labels", None)
            
            outputs = model.generate(**inputs, max_new_tokens=10)
            input_len = inputs["input_ids"].shape[1]
            pred_text = manager.processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
            
            is_correct = match_answer(pred_text, target)
            if is_correct:
                correct += 1
            total += 1
            
            log_entries.append({
                "index": i + 1,
                "question": prompt,
                "ground_truth": target,
                "model_prediction": pred_text,
                "is_correct": is_correct
            })
            
            if i < 5:
                print(f"   [Mẫu {i+1}] Hỏi: {prompt[:80]}... | Đúng: {target} | Đoán: {pred_text} | Kết quả: {'ĐÚNG' if is_correct else 'SAI'}")
            
    accuracy = (correct / total) * 100 if total > 0 else 0.0
    print(f"[{name}] Accuracy: {accuracy:.2f}% ({correct}/{total})")
    
    name_slug = re.sub(r'[^a-zA-Z0-9_]', '_', name.lower().strip())
    log_path = os.path.join(eval_config["output_dir"], f"eval_details_{name_slug}.json")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as lf:
        json.dump(log_entries, lf, ensure_ascii=False, indent=2)
    print(f"   ➔ Đã lưu nhật ký chi tiết của {name} tại: {log_path}")
    
    return accuracy

print("=== BẮT ĐẦU ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH ===")
results = {}

if cv_dataset is not None:
    test_data = cv_dataset['test']
    num_test = eval_config['eval_datasets']['cv_bench'].get('test_samples', 100)
    test_data_slice = test_data.select(range(max(0, len(test_data) - num_test), len(test_data)))
    
    model.livr_stage = 2
    acc_stage2 = evaluate_accuracy(model, test_data_slice, manager, name="CV-Bench Stage 2 (Mở mắt)", max_samples=num_test)
    
    model.livr_stage = 1
    acc_stage1 = evaluate_accuracy(model, test_data_slice, manager, name="CV-Bench Stage 1 (Bịt mắt - Sanity Check)", max_samples=num_test)
    
    results["CV-Bench"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1
    }

if mathvista_dataset is not None:
    num_test = eval_config['eval_datasets']['math_vista'].get('test_samples', 100)
    test_data_slice = mathvista_dataset.select(range(max(0, len(mathvista_dataset) - num_test), len(mathvista_dataset)))
    
    model.livr_stage = 2
    acc_stage2 = evaluate_accuracy(model, test_data_slice, manager, name="MathVista Stage 2 (Mở mắt)", max_samples=num_test)
    
    model.livr_stage = 1
    acc_stage1 = evaluate_accuracy(model, test_data_slice, manager, name="MathVista Stage 1 (Bịt mắt - Sanity Check)", max_samples=num_test)
    
    results["MathVista"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1
    }

print("\n" + "="*70)
print(" BẢNG TỔNG KẾT HIỆU NĂNG & KIỂM ĐỊNH KHOA HỌC (SANITY CHECK)")
print("="*70)
print(f"{'Dataset':<15} | {'Stage 2 (Mở)':<15} | {'Stage 1 (Bịt)':<20} | {'Sụt giảm':<10}")
print("-"*70)
for ds_name, metrics in results.items():
    print(f"{ds_name:<15} | {metrics['Stage 2 (Mở)']:>13.2f}% | {metrics['Stage 1 (Bịt - Sanity Check)']:>18.2f}% | {metrics['Sụt giảm']:>8.2f}%")
print("="*70)
print("Giải nghĩa khoa học:")
print("1. Stage 2 (Mở mắt): Đo lường khả năng giải quyết tác vụ khi ảnh hiển thị đầy đủ.")
print("2. Stage 1 (Bịt mắt): Chặn ảnh hoàn toàn. Mô hình bắt buộc phải trả lời dựa trên thông tin tích lũy")
print("   trong Latent Tokens.")
print("3. Mức sụt giảm vừa phải chứng minh Latent Tokens đóng vai trò là một 'hộp đen' thị giác xuất sắc!")

=== BẮT ĐẦU ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH ===

➔ Đang chạy đánh giá trên CV-Bench Stage 2 (Mở mắt) (Giới hạn 100 mẫu)...
   [Mẫu 1] Hỏi: Estimate the real-world distances between objects in this image. Which object is... | Đúng: (A) | Đoán: (A) | Kết quả: ĐÚNG
   [Mẫu 2] Hỏi: Estimate the real-world distances between objects in this image. Which object is... | Đúng: (B) | Đoán: (B) | Kết quả: ĐÚNG
   [Mẫu 3] Hỏi: Estimate the real-world distances between objects in this image. Which object is... | Đúng: (A) | Đoán: (B) | Kết quả: SAI
   [Mẫu 4] Hỏi: Estimate the real-world distances between objects in this image. Which object is... | Đúng: (B) | Đoán: (B) | Kết quả: ĐÚNG
   [Mẫu 5] Hỏi: Estimate the real-world distances between objects in this image. Which object is... | Đúng: (B) | Đoán: (B) | Kết quả: ĐÚNG
[CV-Bench Stage 2 (Mở mắt)] Accuracy: 68.00% (68/100)
   ➔ Đã lưu nhật ký chi tiết của CV-Bench Stage 2 (Mở mắt) tại: /kaggle/working/checkpoints/evaluation/eval_details_cv_bench_sta